# Airflow 3: Backfill data intervals

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/airflow/airflow-03-backfill.ipynb)

Exercise historical Airflow data intervals with local runs and check the published partitions.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q apache-airflow~=3.4

## Setup and sample data

Create a local Airflow metadata database. The fixture covers three historical days.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory()
home = Path(workspace.name)
dags = home / "dags"
dags.mkdir()
env = {**os.environ, "AIRFLOW_HOME": str(home), "AIRFLOW__CORE__LOAD_EXAMPLES": "False"}


def run_local(*args):
    run = subprocess.run([sys.executable, *args], env=env, capture_output=True, text=True)
    if run.returncode:
        raise RuntimeError(run.stderr or run.stdout)
    for line in run.stdout.splitlines():
        if line.startswith("LAB "):
            print(line)


run_local("-m", "airflow", "db", "migrate")

## Run and replay historical intervals

A backfill reruns work for historical data intervals. Use the interval start and end to select input and a stable partition name to publish the output. This lab issues local dag.test runs for three historical trigger times, then repeats the middle date. With the daily interval timetable, a run triggered at midnight processes the day that just ended.

**Where it matters**

Verify a historical run and its retry before submitting a backfill to the scheduler.

**Watch out for**

A local test does not exercise scheduler backfill concurrency or reprocessing policies. In deployment, use airflow backfill create with a bounded date range and inspect its dry run first.

In [ ]:
file = dags / "backfill_lab.py"
file.write_text("""
from datetime import datetime, timezone
from pathlib import Path
from airflow.sdk import dag, get_current_context, task
from airflow.timetables.interval import CronDataIntervalTimetable

output = Path(__file__).resolve().parent.parent / "published"

@dag(schedule=CronDataIntervalTimetable("0 0 * * *", timezone="UTC"), start_date=datetime(2026, 2, 1, tzinfo=timezone.utc))
def daily_orders():
    @task
    def publish():
        context = get_current_context()
        day = context["data_interval_start"].date()
        output.mkdir(exist_ok=True)
        (output / f"{day}.csv").write_text(f"date,cents\\n{day},1200\\n")
        print("LAB interval:", day, context["data_interval_end"].date())

    publish()

pipeline = daily_orders()

if __name__ == "__main__":
    for day in (2, 3, 4, 3):
        run = pipeline.test(logical_date=datetime(2026, 2, day, tzinfo=timezone.utc))
        print("LAB state:", run.state)
    print("LAB published files:", sorted(path.name for path in output.glob("*.csv")))
""")
run_local(str(file))

**What to notice:** Four successful runs leave three date-partition files. The repeated February 2 interval rewrites the same file.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

What would break if each run selected data using the current date?

In [ ]:
# Try your solution here.

In [ ]:
print("Historical runs would process today’s partition instead of their assigned intervals.")

## Key takeaways

- Verify a historical run and its retry before submitting a backfill to the scheduler.

**Next:** [Change data capture](../concepts/change-data-capture.ipynb)